In [1]:
import os
os.listdir(); os.chdir("/home/azm0269@auburn.edu/clover/")

from clover.utils.utils import notebook_line_magic
notebook_line_magic()


# DDPO: denoising diffusion policy optimization

This notebook implements DDPO-style policy optimization for a text-to-image diffusion model. The denoising process is treated as a trajectory, each denoising transition receives a policy log probability, and a black-box terminal image reward is optimized with a PPO clipped objective.

The default reward below is intentionally lightweight so the notebook runs without a separate reward model. Replace `reward_fn` with an aesthetic, CLIP, ImageReward, human preference, or task-specific scorer when you are ready to train against a real objective.


In [ ]:
from __future__ import annotations

import gc
import math
import random
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Callable

import numpy as np
import torch
import torch.nn.functional as F
from diffusers import DDPMScheduler, StableDiffusionPipeline
from PIL import Image
from torch import Tensor
from tqdm.auto import trange


In [ ]:
@dataclass
class DDPOConfig:
    model_id: str = "runwayml/stable-diffusion-v1-5"
    output_dir: str = "outputs/ddpo"
    seed: int = 17

    prompt: str = "a colorful clover field at sunrise, high detail"
    negative_prompt: str = "blurry, low quality, distorted"
    train_prompts: tuple[str, ...] = (
        "a colorful clover field at sunrise, high detail",
        "a close-up photo of a bright green clover leaf with dew",
        "a small robot holding a clover in a clean studio photo",
        "an impressionist painting of clovers under warm sunlight",
    )

    height: int = 512
    width: int = 512
    num_inference_steps: int = 30
    guidance_scale: float = 7.5
    eta: float = 1.0

    rollouts_per_epoch: int = 4
    train_epochs: int = 10
    ppo_epochs: int = 4
    minibatch_size: int = 2
    learning_rate: float = 1e-6
    clip_range: float = 1e-4
    max_grad_norm: float = 1.0
    mixed_precision: bool = True
    gradient_checkpointing: bool = True

    log_every: int = 1
    save_every: int = 5


cfg = DDPOConfig()
Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)
cfg


In [ ]:
def set_seed(seed: int) -> torch.Generator:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    return torch.Generator(device=device).manual_seed(seed)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float16 if device.type == "cuda" and cfg.mixed_precision else torch.float32
generator = set_seed(cfg.seed)
print(device, dtype)


In [ ]:
@torch.no_grad()
def aesthetic_proxy_reward(images: list[Image.Image], prompts: list[str]) -> Tensor:
    """Small offline reward used as a placeholder for a real preference model.

    It rewards images that are bright, saturated, and have moderate contrast. DDPO only
    needs terminal scalar rewards, so any black-box scorer with this signature can be
    swapped in here.
    """
    rewards = []
    for image in images:
        arr = torch.from_numpy(np.asarray(image).astype("float32") / 255.0)
        mean_rgb = arr.mean(dim=(0, 1))
        brightness = arr.mean()
        saturation = (mean_rgb.max() - mean_rgb.min()).clamp_min(0.0)
        contrast = arr.std().clamp_max(0.35)
        rewards.append(0.45 * brightness + 0.35 * saturation + 0.20 * contrast)
    return torch.stack(rewards).to(device)


reward_fn: Callable[[list[Image.Image], list[str]], Tensor] = aesthetic_proxy_reward


In [ ]:
def load_pipeline(config: DDPOConfig) -> StableDiffusionPipeline:
    pipe = StableDiffusionPipeline.from_pretrained(
        config.model_id,
        torch_dtype=dtype,
        safety_checker=None,
        requires_safety_checker=False,
    )
    pipe.scheduler = DDPMScheduler.from_config(pipe.scheduler.config)
    pipe = pipe.to(device)

    pipe.vae.requires_grad_(False)
    pipe.text_encoder.requires_grad_(False)
    pipe.unet.train()
    pipe.unet.requires_grad_(True)

    if config.gradient_checkpointing and hasattr(pipe.unet, "enable_gradient_checkpointing"):
        pipe.unet.enable_gradient_checkpointing()

    try:
        pipe.enable_xformers_memory_efficient_attention()
        print("xFormers memory efficient attention enabled")
    except Exception as exc:
        print(f"xFormers not enabled: {exc}")

    return pipe


pipe = load_pipeline(cfg)
optimizer = torch.optim.AdamW(pipe.unet.parameters(), lr=cfg.learning_rate)
vae_scale_factor = 2 ** (len(pipe.vae.config.block_out_channels) - 1)


In [ ]:
def encode_prompts(pipe: StableDiffusionPipeline, prompts: list[str], negative_prompt: str) -> Tensor:
    tokenized = pipe.tokenizer(
        prompts,
        padding="max_length",
        max_length=pipe.tokenizer.model_max_length,
        truncation=True,
        return_tensors="pt",
    )
    negative = pipe.tokenizer(
        [negative_prompt] * len(prompts),
        padding="max_length",
        max_length=pipe.tokenizer.model_max_length,
        truncation=True,
        return_tensors="pt",
    )
    with torch.no_grad():
        text_embeds = pipe.text_encoder(tokenized.input_ids.to(device))[0]
        negative_embeds = pipe.text_encoder(negative.input_ids.to(device))[0]
    return torch.cat([negative_embeds, text_embeds], dim=0).to(dtype=dtype)


def predict_noise(pipe: StableDiffusionPipeline, latents: Tensor, timestep: Tensor | int, prompt_embeds: Tensor) -> Tensor:
    latent_model_input = torch.cat([latents, latents], dim=0)
    noise_pred = pipe.unet(latent_model_input, timestep, encoder_hidden_states=prompt_embeds).sample
    noise_uncond, noise_text = noise_pred.chunk(2)
    return noise_uncond + cfg.guidance_scale * (noise_text - noise_uncond)


def decode_latents(pipe: StableDiffusionPipeline, latents: Tensor) -> list[Image.Image]:
    latents = latents / pipe.vae.config.scaling_factor
    with torch.no_grad():
        images = pipe.vae.decode(latents.to(dtype=pipe.vae.dtype)).sample
    images = (images / 2 + 0.5).clamp(0, 1)
    images = images.detach().cpu().permute(0, 2, 3, 1).float().numpy()
    images = (images * 255).round().astype("uint8")
    return [Image.fromarray(image) for image in images]


In [ ]:
def previous_timestep(scheduler: DDPMScheduler, timestep: int) -> int:
    if hasattr(scheduler, "previous_timestep"):
        prev = scheduler.previous_timestep(timestep)
        return int(prev.item() if torch.is_tensor(prev) else prev)
    step = scheduler.config.num_train_timesteps // scheduler.num_inference_steps
    return int(timestep) - step


def ddpm_mean_std(scheduler: DDPMScheduler, model_output: Tensor, timestep: int, sample: Tensor) -> tuple[Tensor, Tensor]:
    """Mean and std of p_theta(x_{t-1} | x_t) for fixed-variance DDPM transitions."""
    if model_output.shape[1] == sample.shape[1] * 2:
        model_output, _ = torch.split(model_output, sample.shape[1], dim=1)

    t = int(timestep)
    prev_t = previous_timestep(scheduler, t)
    alphas_cumprod = scheduler.alphas_cumprod.to(device=sample.device, dtype=sample.dtype)
    alpha_prod_t = alphas_cumprod[t]
    alpha_prod_t_prev = alphas_cumprod[prev_t] if prev_t >= 0 else scheduler.final_alpha_cumprod.to(sample.device, sample.dtype)
    beta_prod_t = 1 - alpha_prod_t
    beta_prod_t_prev = 1 - alpha_prod_t_prev

    prediction_type = getattr(scheduler.config, "prediction_type", "epsilon")
    if prediction_type == "epsilon":
        pred_original_sample = (sample - beta_prod_t.sqrt() * model_output) / alpha_prod_t.sqrt()
    elif prediction_type == "sample":
        pred_original_sample = model_output
    elif prediction_type == "v_prediction":
        pred_original_sample = alpha_prod_t.sqrt() * sample - beta_prod_t.sqrt() * model_output
    else:
        raise ValueError(f"Unsupported prediction type: {prediction_type}")

    if scheduler.config.clip_sample:
        clip_range = getattr(scheduler.config, "clip_sample_range", 1.0)
        pred_original_sample = pred_original_sample.clamp(-clip_range, clip_range)

    current_alpha_t = alpha_prod_t / alpha_prod_t_prev
    current_beta_t = 1 - current_alpha_t
    pred_original_sample_coeff = alpha_prod_t_prev.sqrt() * current_beta_t / beta_prod_t
    current_sample_coeff = current_alpha_t.sqrt() * beta_prod_t_prev / beta_prod_t
    mean = pred_original_sample_coeff * pred_original_sample + current_sample_coeff * sample

    variance = (beta_prod_t_prev / beta_prod_t) * current_beta_t
    variance = variance.clamp(min=1e-20)
    std = (cfg.eta * variance.sqrt()).to(sample.dtype)
    return mean, std


def transition_log_prob(mean: Tensor, std: Tensor, prev_sample: Tensor) -> Tensor:
    log_prob = -0.5 * ((prev_sample - mean) / std).pow(2) - torch.log(std) - 0.5 * math.log(2 * math.pi)
    return log_prob.flatten(1).mean(dim=1)


def ddpm_step_with_log_prob(
    scheduler: DDPMScheduler,
    model_output: Tensor,
    timestep: int,
    sample: Tensor,
    generator: torch.Generator | None = None,
    prev_sample: Tensor | None = None,
) -> tuple[Tensor, Tensor]:
    mean, std = ddpm_mean_std(scheduler, model_output, timestep, sample)
    if prev_sample is None:
        noise = torch.randn(sample.shape, generator=generator, device=sample.device, dtype=sample.dtype)
        prev_sample = mean + std * noise
    log_prob = transition_log_prob(mean, std, prev_sample)
    return prev_sample, log_prob


In [ ]:
def sample_prompt_batch(batch_size: int) -> list[str]:
    return [random.choice(cfg.train_prompts) for _ in range(batch_size)]


@torch.no_grad()
def collect_rollouts(pipe: StableDiffusionPipeline, batch_size: int) -> dict[str, Tensor | list[str] | list[Image.Image]]:
    pipe.unet.eval()
    prompts = sample_prompt_batch(batch_size)
    prompt_embeds = encode_prompts(pipe, prompts, cfg.negative_prompt)
    pipe.scheduler.set_timesteps(cfg.num_inference_steps, device=device)

    latent_shape = (
        batch_size,
        pipe.unet.config.in_channels,
        cfg.height // vae_scale_factor,
        cfg.width // vae_scale_factor,
    )
    latents = torch.randn(latent_shape, generator=generator, device=device, dtype=dtype)
    latents = latents * pipe.scheduler.init_noise_sigma

    states, actions, log_probs, timesteps = [], [], [], []
    for t in pipe.scheduler.timesteps:
        timestep = int(t.item())
        states.append(latents.detach().float().cpu())
        noise_pred = predict_noise(pipe, latents, t, prompt_embeds)
        next_latents, log_prob = ddpm_step_with_log_prob(pipe.scheduler, noise_pred, timestep, latents, generator)
        actions.append(next_latents.detach().float().cpu())
        log_probs.append(log_prob.detach().float().cpu())
        timesteps.append(timestep)
        latents = next_latents

    images = decode_latents(pipe, latents)
    rewards = reward_fn(images, prompts).detach().float().cpu()
    pipe.unet.train()

    return {
        "prompts": prompts,
        "states": torch.stack(states, dim=1),
        "actions": torch.stack(actions, dim=1),
        "old_log_probs": torch.stack(log_probs, dim=1),
        "timesteps": torch.tensor(timesteps, dtype=torch.long),
        "rewards": rewards,
        "images": images,
    }


In [ ]:
def normalize_advantages(rewards: Tensor) -> Tensor:
    advantages = rewards - rewards.mean()
    if rewards.numel() > 1:
        advantages = advantages / (rewards.std(unbiased=False) + 1e-8)
    return advantages


def ppo_update(pipe: StableDiffusionPipeline, rollout: dict[str, Tensor | list[str]], optimizer: torch.optim.Optimizer) -> dict[str, float]:
    states = rollout["states"]
    actions = rollout["actions"]
    old_log_probs = rollout["old_log_probs"]
    timesteps = rollout["timesteps"].tolist()
    rewards = rollout["rewards"]
    prompts = rollout["prompts"]
    advantages = normalize_advantages(rewards).to(device)

    batch_size, trajectory_len = old_log_probs.shape
    indices = torch.arange(batch_size)
    losses, approx_kls, clip_fracs = [], [], []

    pipe.unet.train()
    for _ in range(cfg.ppo_epochs):
        permutation = indices[torch.randperm(batch_size)]
        for start in range(0, batch_size, cfg.minibatch_size):
            mb_idx = permutation[start:start + cfg.minibatch_size]
            mb_prompts = [prompts[i] for i in mb_idx.tolist()]
            prompt_embeds = encode_prompts(pipe, mb_prompts, cfg.negative_prompt)
            mb_advantages = advantages[mb_idx]

            current_log_probs = []
            for step_idx, timestep in enumerate(timesteps):
                t = torch.tensor(timestep, device=device, dtype=torch.long)
                state = states[mb_idx, step_idx].to(device=device, dtype=dtype)
                action = actions[mb_idx, step_idx].to(device=device, dtype=dtype)
                noise_pred = predict_noise(pipe, state, t, prompt_embeds)
                _, log_prob = ddpm_step_with_log_prob(pipe.scheduler, noise_pred, timestep, state, prev_sample=action)
                current_log_probs.append(log_prob)

            current_log_probs = torch.stack(current_log_probs, dim=1)
            old = old_log_probs[mb_idx].to(device=device, dtype=current_log_probs.dtype)
            adv = mb_advantages[:, None].expand_as(current_log_probs)

            log_ratio = current_log_probs - old
            ratio = torch.exp(log_ratio.clamp(-10, 10))
            unclipped = ratio * adv
            clipped = torch.clamp(ratio, 1 - cfg.clip_range, 1 + cfg.clip_range) * adv
            loss = -torch.min(unclipped, clipped).mean()

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(pipe.unet.parameters(), cfg.max_grad_norm)
            optimizer.step()

            with torch.no_grad():
                approx_kl = (old - current_log_probs).mean().abs()
                clip_frac = ((ratio - 1).abs() > cfg.clip_range).float().mean()
            losses.append(float(loss.detach().cpu()))
            approx_kls.append(float(approx_kl.detach().cpu()))
            clip_fracs.append(float(clip_frac.detach().cpu()))

    return {
        "loss": float(np.mean(losses)),
        "approx_kl": float(np.mean(approx_kls)),
        "clip_frac": float(np.mean(clip_fracs)),
        "reward_mean": float(rewards.mean()),
        "reward_std": float(rewards.std(unbiased=False)) if rewards.numel() > 1 else 0.0,
    }


In [ ]:
history = []
for epoch in trange(1, cfg.train_epochs + 1):
    rollout = collect_rollouts(pipe, cfg.rollouts_per_epoch)
    metrics = ppo_update(pipe, rollout, optimizer)
    metrics["epoch"] = epoch
    history.append(metrics)

    if epoch % cfg.log_every == 0:
        print(metrics)

    if epoch % cfg.save_every == 0:
        ckpt_dir = Path(cfg.output_dir) / f"unet_epoch_{epoch:04d}"
        pipe.unet.save_pretrained(ckpt_dir)
        for i, image in enumerate(rollout["images"]):
            image.save(Path(cfg.output_dir) / f"epoch_{epoch:04d}_sample_{i:02d}.png")

    del rollout
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()


In [ ]:
@torch.no_grad()
def generate_eval_images(pipe: StableDiffusionPipeline, prompts: list[str], seed: int = 123) -> list[Image.Image]:
    pipe.unet.eval()
    eval_generator = torch.Generator(device=device).manual_seed(seed)
    images = pipe(
        prompts,
        negative_prompt=[cfg.negative_prompt] * len(prompts),
        height=cfg.height,
        width=cfg.width,
        num_inference_steps=cfg.num_inference_steps,
        guidance_scale=cfg.guidance_scale,
        generator=eval_generator,
    ).images
    pipe.unet.train()
    return images


eval_prompts = [cfg.prompt, *cfg.train_prompts[:3]]
eval_images = generate_eval_images(pipe, eval_prompts)
for i, image in enumerate(eval_images):
    image.save(Path(cfg.output_dir) / f"eval_{i:02d}.png")

eval_images[0]


In [ ]:
final_dir = Path(cfg.output_dir) / "unet_final"
pipe.unet.save_pretrained(final_dir)
print(f"Saved fine-tuned UNet to {final_dir}")
print(asdict(cfg))
